## Embedding
This notebook covers chapter _2. Working with text data_.


#### Tokenization

chapter 2.1 - 2.5, pages 17 - 35

In this section, OpenAI's byte pair encoding (BPE) tokenizer _ticktoken_ is used to tokenize text.


In [1]:
import torch
import tiktoken
from importlib.metadata import version

print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [2]:
tokenizer = tiktoken.get_encoding("gpt2")

In [3]:
text = "Hello, how are you? <|endoftext|> I like superkaligrafilistischexpiallegorisch."
tokens_ids = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(tokens_ids)

[15496, 11, 703, 389, 345, 30, 220, 50256, 314, 588, 2208, 74, 282, 328, 430, 10379, 396, 2304, 33095, 79, 498, 1455, 273, 25308, 13]


In [4]:
tokenizer.decode(tokens_ids)

'Hello, how are you? <|endoftext|> I like superkaligrafilistischexpiallegorisch.'

Exercise 2.1

In [5]:
input = "Akwirw ier"
print("Input: ", input)
tokens_ids = tokenizer.encode(input);
print("Encoded tokens: ",tokens_ids)

Input:  Akwirw ier
Encoded tokens:  [33901, 86, 343, 86, 220, 959]


In [6]:
token0 = tokenizer.decode([33901])
token1 = tokenizer.decode([86])
token2 = tokenizer.decode([343])
token3 = tokenizer.decode([86])
token4 = tokenizer.decode([220])
token5 = tokenizer.decode([959])

print("Decoded tokens: ", token0, token1, token2, token3, token4, token5, sep=" | ")

Decoded tokens:  | Ak | w | ir | w |   | ier


#### Data sampling with a sliding window

Chapter 2.6, pages 35 - 41

In [7]:
# Create ticktoken BPE tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

In [8]:
# Read text file
SENTENCES_FILE_PATH = "../data/the-verdict.txt"

with open(SENTENCES_FILE_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [9]:
# Tokenize text
txt = tokenizer.encode(raw_text)
print("Number of tokens: ", len(txt))

Number of tokens:  5145


In [10]:
# Take a sample
sample = txt[1010:1055]
print(f"Token IDs: ", sample)
print(f"Tokens: ")

for token_id in sample:
    print(tokenizer.decode([token_id]), end="|")

Token IDs:  [35924, 262, 12306, 395, 4133, 13, 198, 198, 1, 26788, 338, 691, 12226, 318, 284, 1234, 8737, 656, 19133, 553, 373, 530, 286, 262, 7877, 72, 3150, 339, 8104, 866, 1973, 262, 37918, 411, 290, 8465, 286, 281, 33954, 271, 3973, 9899, 14678, 40556, 12]
Tokens: 
poke| the| ampl|est| resources|.|
|
|"|Money|'s| only| excuse| is| to| put| beauty| into| circulation|,"| was| one| of| the| ax|i|oms| he| laid| down| across| the| Sev|res| and| silver| of| an| exqu|is|itely| appointed| lun|cheon|-|

In [11]:
context_size = 4 # How many tokens do we look at when predicting the next token?
x = txt[:context_size] # Input tokens
y = txt[1:context_size+1] # Target token (the one we want to predict)
print("x: ", x)
print("y:     ", y)

x:  [40, 367, 2885, 1464]
y:      [367, 2885, 1464, 1807]


In [12]:
print("[input token IDs] -> target token ID")
print("-------------------------------")

for i in range(1, context_size+1):
    context = sample[:i]
    desired = sample[i]
    print(f"{context} -> {desired}")

[input token IDs] -> target token ID
-------------------------------
[35924] -> 262
[35924, 262] -> 12306
[35924, 262, 12306] -> 395
[35924, 262, 12306, 395] -> 4133


In [13]:
print("[input tokens] -> target token")
print("-------------------------------")

for i in range(1, context_size+1):
    context = sample[:i]
    desired = sample[i]
    print(f"{tokenizer.decode(context)} -> {tokenizer.decode([desired])}")

[input tokens] -> target token
-------------------------------
poke ->  the
poke the ->  ampl
poke the ampl -> est
poke the amplest ->  resources


In [14]:
import torch
from torch.utils.data import Dataset, DataLoader

class GPTDataset1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        token_ids = tokenizer.encode(txt)
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i+max_length]
            target_chunk = token_ids[i+1:i+max_length+1]

            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        """Returns the number of samples in the dataset."""
        return len(self.input_ids)

    def __getitem__(self, idx):
        """Returns a single sample from the dataset."""
        return self.input_ids[idx], self.target_ids[idx]

In [15]:
# Demonstration of GPTDataset1 with dummy data.
example_token_ids = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

max_length = 4
stride = 1

print("[input token IDs chunk] -> [target token IDs chunk]")
print("---------------------------------------")

for i in range(0, len(example_token_ids) - max_length, stride):
    input_chunk = example_token_ids[i:i+max_length]
    target_chunk = example_token_ids[i+1:i+max_length+1]
    print(f"{input_chunk} -> {target_chunk}")

[input token IDs chunk] -> [target token IDs chunk]
---------------------------------------
[0, 1, 2, 3] -> [1, 2, 3, 4]
[1, 2, 3, 4] -> [2, 3, 4, 5]
[2, 3, 4, 5] -> [3, 4, 5, 6]
[3, 4, 5, 6] -> [4, 5, 6, 7]
[4, 5, 6, 7] -> [5, 6, 7, 8]
[5, 6, 7, 8] -> [6, 7, 8, 9]
[6, 7, 8, 9] -> [7, 8, 9, 10]
[7, 8, 9, 10] -> [8, 9, 10, 11]
[8, 9, 10, 11] -> [9, 10, 11, 12]
[9, 10, 11, 12] -> [10, 11, 12, 13]
[10, 11, 12, 13] -> [11, 12, 13, 14]
[11, 12, 13, 14] -> [12, 13, 14, 15]
[12, 13, 14, 15] -> [13, 14, 15, 16]
[13, 14, 15, 16] -> [14, 15, 16, 17]
[14, 15, 16, 17] -> [15, 16, 17, 18]
[15, 16, 17, 18] -> [16, 17, 18, 19]


In [16]:
def create_dataloader_v1(txt: str,
                         batch_size: int = 4,
                         max_length: int = 256,
                         stride: int = 128,
                         shuffle: bool = True,
                         drop_last: bool = True,
                         num_workers: int = 0):
    """Creates a torch.utilsdata.DataLoader
    
    Args:
        txt: The input text to be tokenized and used for training.
        batch_size: The number of samples per batch.
        max_length: The maximum length of input sequences.
        stride: The step size for creating overlapping sequences.
        shuffle: Whether to shuffle the dataset.
        drop_last : Whether to drop the last batch if it is shorter than batch_size. Prevents loss spikes during training.
        num_workers: The number of subprocesses to use for preprocessing.

        Returns:    A DataLoader that yields batches of (input_ids, target_ids) tuples.
    """

    tokenizer = tiktoken.get_encoding("gpt2")
    dataset = GPTDataset1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last =drop_last,
        num_workers=num_workers
    )

    return dataloader

In [17]:
with open(SENTENCES_FILE_PATH, "r", encoding="utf-8") as f:
    raw_text = f.read()

    dataloader = create_dataloader_v1(txt=raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
    data_iter = iter(dataloader)
    first_batch = next(data_iter)

In [18]:
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [19]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


In [20]:
dataloader2 = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

In [21]:
data_iter2 = iter(dataloader2)
inputs, targets = next(data_iter2)
print("Inputs:\n", inputs)
print("Targets:\n", targets)

Inputs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
Targets:
 tensor([[  367,  2885,  1464,  1807],
        [ 3619,   402,   271, 10899],
        [ 2138,   257,  7026, 15632],
        [  438,  2016,   257,   922],
        [ 5891,  1576,   438,   568],
        [  340,   373,   645,  1049],
        [ 5975,   284,   502,   284],
        [ 3285,   326,    11,   287]])


#### Token Embeddings

Chapter 2.7 & 2.8, pages 41 - 49

In [37]:
# Create a simplified embedding layer

input_ids= torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print("Embedding layer: ")
print(embedding_layer.weight)


Embedding layer: 
Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [39]:
# Apply embedding layer to one input token ID
print("Embedding for token ID 3:\n", embedding_layer(torch.tensor([3])))

Embedding for token ID 3:
 tensor([[-0.4015,  0.9666, -1.1481]], grad_fn=<EmbeddingBackward0>)


In [40]:
# Apply embedding layer to all our example input token IDs
print("Embeddings for all input token IDs:\n", embedding_layer(input_ids))

Embeddings for all input token IDs:
 tensor([[ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-2.8400, -0.7849, -1.4096],
        [ 0.9178,  1.5810,  1.3010]], grad_fn=<EmbeddingBackward0>)


#### Encoding word positions

- The same token is always mapped to the same embedding
- The order of the words in a sequence is not considered
- Injecting additional position information into the embeddings improves the LLM
- There are two categories of position-aware embeddings:
    1. Relative positional embeddings
        - For each position in the input sequence, a unique embedding is added to the token's embedding
        - Used by OpenAI's GPT models
    2. Absolute position embeddings
        - Adds distance between token's to the embeddings
        - Advanteage: Model can generalize better to sequences of varying lengths

(see Figure 2.18 on page 45)

Now, lets create the embedding layer for the whole _The Verdict_ story:

In [48]:
# Create token embedding layer
vocab_size = 50257 # Number of tokens in GPT-2's vocabulary
output_dim = 256 # Hyperparameter
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

print("Token embedding layer's shape:\n", token_embedding_layer.weight.shape)
print("\nToken embedding layer:\n", token_embedding_layer.weight)

Token embedding layer's shape:
 torch.Size([50257, 256])

Token embedding layer:
 Parameter containing:
tensor([[-0.0534, -1.3168,  1.0696,  ...,  0.1199, -0.6286, -0.8631],
        [-1.8645,  0.8835, -1.6282,  ..., -1.2460, -0.4325, -0.9073],
        [-1.2796,  0.6160, -1.0969,  ..., -0.9785,  0.7252, -0.4517],
        ...,
        [ 0.1519, -0.4437, -0.2233,  ..., -0.4150,  0.9106,  0.6115],
        [-0.6259, -0.6180, -0.1673,  ..., -0.2670, -0.0824,  1.3767],
        [ 0.0341,  0.0150, -2.1869,  ..., -0.2971, -1.1700, -0.3738]],
       requires_grad=True)


In [43]:
# Create data loader
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False)
dater_iter = iter(dataloader)
inputs, targets = next(dater_iter)
print("Input token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

Input token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [50]:
# Use the embedding layer to convert input token IDs to embeddings
token_embeddings = token_embedding_layer(inputs)
print("\nToken embeddings shape:\n", token_embeddings.shape)


Token embeddings shape:
 torch.Size([8, 4, 256])


A batch contains 8 sequences, each consisting of 4 tokens, and each token is represented by a 256-dimensional embedding vector. Therefore shape 8 x 4 x 256.

Next, lets creat the positional embeddings.

- In the original transformer, the positional encodings were fixed and predefined.
- For example [0, ..., 0] for the first token, [0, ..., 1] for the second, and so on.
- OpenAI's GPT models optimize the positional embeddings during training, instead of using fixed ones.
- We will now create the initial positional embeddings that will later be optimized.

In [53]:
# Create positional embeddings
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)
pos_embeddings = pos_embedding_layer(torch.arange(context_length))
print("\nPositional embeddings:\n", pos_embeddings)
print("Shape:\n", pos_embeddings.shape)


Positional embeddings:
 tensor([[ 0.9125,  0.7239, -0.1887,  ...,  0.7097, -1.9931, -1.5533],
        [ 0.2822,  0.4822, -1.7930,  ...,  1.0858,  0.9021, -1.2709],
        [ 0.3706, -0.5216,  0.8292,  ..., -0.9160,  1.7922, -0.7494],
        [-0.2343, -0.7527,  0.6884,  ..., -1.1314, -0.7565,  0.3741]],
       grad_fn=<EmbeddingBackward0>)
Shape:
 torch.Size([4, 256])


The length of a sequence is 4. Therefore, we need 4 positional embeddings, one for each position. Since the dimension of the embeddings is 256, the dimension of the posititonal embeddings must be 256, too.

In the cell below, we will combine both embeddings, to get the fully embedded token IDs, that we can use as input for the LLM.

In [54]:
# Add positional embeddings tensor to token embeddings tensor
input_embeddings = token_embeddings + pos_embeddings
print("\nInput embeddings shape:\n", input_embeddings.shape)


Input embeddings shape:
 torch.Size([8, 4, 256])
